[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C01_LLM_Internals_Course/03_train_minigpt/03_train_minigpt.ipynb)

# 03 · 从零训练 mini-GPT —— 配套 Notebook

**CPU 可跑 | 训练全程约 3–5 分钟 | 纯 PyTorch，零外部数据 / 零下载**

本 notebook 把模块 02 手写的 Transformer 架构真正"点火"训练起来。完整路径：

1. 内嵌公版 Shakespeare 选段语料 + **字符级 tokenizer**（模块 01 的 BPE 可无缝替换）
2. **MiniGPT**（复用模块 02 的架构定义：`n_layer=2, n_head=4, n_embd=64, block_size=64`）+ GPT-2 式初始化（残差投影 $1/\sqrt{2N}$ 缩放）
3. **sanity check**：初始 loss $\approx \ln(\text{vocab\_size})$（assert）
4. 完整训练循环 ~1000 步：**AdamW + 线性 warmup / cosine 衰减 + gradient clipping**
5. 训练动力学：train/val loss 曲线 + 采样质量演变（乱码 → 词形 → 局部连贯 → 风格）
6. **过拟合实验**：数据砍到 1/10 再训，观察 train/val 分叉
7. ✏️ 3 道练习：`get_batch` / `lr_schedule` / perplexity 换算

对应讲解：`03_讲解.html`。论文锚点：GPT-2 [Radford 2019]、GPT-3 [Brown 2020]、AdamW [Loshchilov 2017]。

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# 模块 02 的小配置：参数量 ~0.1M，CPU 上 1000 步约 1–3 分钟
cfg = dict(
    n_layer=2,      # Transformer block 数（N）
    n_head=4,       # attention 头数
    n_embd=64,      # 残差流宽度 d_model
    block_size=64,  # 上下文长度 T
    dropout=0.1,    # 小数据上有用；大规模单 epoch 预训练通常设 0
    batch_size=32,
)
device = 'cpu'
print(torch.__version__, '| device =', device)

## 1 · 数据与字符级 tokenizer

预训练数据管线没有"样本"概念，只有**一条 token 长流**：语料 → tokenizer → 一维 id 序列 → 按 90/10 切 train/val 流 → 每步从 train 流上**随机截窗口**。

这里用**字符级 tokenizer**（`char -> id` 直查表）：词表 ~60、零依赖、保证无 `<unk>`。它与模块 01 的 BPE 接口完全一致（`encode/decode`），把下面两个函数换成 BPE 版本即可升级为 subword 模型——其余代码一行不用改。

语料为内嵌的 Shakespeare 公版选段（约 20KB）。注意 train/val 是**在流的层面**切的：val 是模型从未见过的最后 10%，这是后面一切过拟合诊断的依据。

In [ ]:
# 内嵌公版语料：Shakespeare 选段（sonnets + 著名独白，现代拼写、ASCII）
TEXT = '''
Shall I compare thee to a summer's day?
Thou art more lovely and more temperate:
Rough winds do shake the darling buds of May,
And summer's lease hath all too short a date:
Sometime too hot the eye of heaven shines,
And often is his gold complexion dimm'd;
And every fair from fair sometime declines,
By chance, or nature's changing course, untrimm'd;
But thy eternal summer shall not fade,
Nor lose possession of that fair thou owest;
Nor shall Death brag thou wander'st in his shade,
When in eternal lines to time thou growest;
So long as men can breathe, or eyes can see,
So long lives this, and this gives life to thee.

When, in disgrace with fortune and men's eyes,
I all alone beweep my outcast state,
And trouble deaf heaven with my bootless cries,
And look upon myself, and curse my fate,
Wishing me like to one more rich in hope,
Featured like him, like him with friends possess'd,
Desiring this man's art and that man's scope,
With what I most enjoy contented least;
Yet in these thoughts myself almost despising,
Haply I think on thee, and then my state,
Like to the lark at break of day arising
From sullen earth, sings hymns at heaven's gate;
For thy sweet love remember'd such wealth brings
That then I scorn to change my state with kings.

When to the sessions of sweet silent thought
I summon up remembrance of things past,
I sigh the lack of many a thing I sought,
And with old woes new wail my dear time's waste:
Then can I drown an eye, unused to flow,
For precious friends hid in death's dateless night,
And weep afresh love's long since cancell'd woe,
And moan the expense of many a vanish'd sight:
Then can I grieve at grievances foregone,
And heavily from woe to woe tell o'er
The sad account of fore-bemoaned moan,
Which I new pay as if not paid before.
But if the while I think on thee, dear friend,
All losses are restored and sorrows end.

Like as the waves make towards the pebbled shore,
So do our minutes hasten to their end;
Each changing place with that which goes before,
In sequent toil all forwards do contend.
Nativity, once in the main of light,
Crawls to maturity, wherewith being crown'd,
Crooked eclipses 'gainst his glory fight,
And Time that gave doth now his gift confound.
Time doth transfix the flourish set on youth
And delves the parallels in beauty's brow,
Feeds on the rarities of nature's truth,
And nothing stands but for his scythe to mow:
And yet to times in hope my verse shall stand,
Praising thy worth, despite his cruel hand.

That time of year thou mayst in me behold
When yellow leaves, or none, or few, do hang
Upon those boughs which shake against the cold,
Bare ruin'd choirs, where late the sweet birds sang.
In me thou seest the twilight of such day
As after sunset fadeth in the west,
Which by and by black night doth take away,
Death's second self, that seals up all in rest.
In me thou see'st the glowing of such fire
That on the ashes of his youth doth lie,
As the death-bed whereon it must expire,
Consumed with that which it was nourish'd by.
This thou perceivest, which makes thy love more strong,
To love that well which thou must leave ere long.

Let me not to the marriage of true minds
Admit impediments. Love is not love
Which alters when it alteration finds,
Or bends with the remover to remove:
O no! it is an ever-fixed mark
That looks on tempests and is never shaken;
It is the star to every wandering bark,
Whose worth's unknown, although his height be taken.
Love's not Time's fool, though rosy lips and cheeks
Within his bending sickle's compass come:
Love alters not with his brief hours and weeks,
But bears it out even to the edge of doom.
If this be error and upon me proved,
I never writ, nor no man ever loved.

My mistress' eyes are nothing like the sun;
Coral is far more red than her lips' red;
If snow be white, why then her breasts are dun;
If hairs be wires, black wires grow on her head.
I have seen roses damask'd, red and white,
But no such roses see I in her cheeks;
And in some perfumes is there more delight
Than in the breath that from my mistress reeks.
I love to hear her speak, yet well I know
That music hath a far more pleasing sound;
I grant I never saw a goddess go;
My mistress, when she walks, treads on the ground:
And yet, by heaven, I think my love as rare
As any she belied with false compare.

To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die: to sleep;
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to, 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep: perchance to dream: ay, there's the rub;
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil,
Must give us pause: there's the respect
That makes calamity of so long life;
For who would bear the whips and scorns of time,
The oppressor's wrong, the proud man's contumely,
The pangs of despised love, the law's delay,
The insolence of office and the spurns
That patient merit of the unworthy takes,
When he himself might his quietus make
With a bare bodkin? who would fardels bear,
To grunt and sweat under a weary life,
But that the dread of something after death,
The undiscover'd country from whose bourn
No traveller returns, puzzles the will
And makes us rather bear those ills we have
Than fly to others that we know not of?
Thus conscience does make cowards of us all;
And thus the native hue of resolution
Is sicklied o'er with the pale cast of thought,
And enterprises of great pith and moment
With this regard their currents turn awry,
And lose the name of action.

What a piece of work is a man! How noble in reason,
how infinite in faculty! In form and moving how
express and admirable! In action how like an angel,
in apprehension how like a god! The beauty of the
world. The paragon of animals. And yet, to me,
what is this quintessence of dust? Man delights not
me: no, nor woman neither.

Speak the speech, I pray you, as I pronounced it to
you, trippingly on the tongue: but if you mouth it,
as many of your players do, I had as lief the
town-crier spoke my lines. Nor do not saw the air
too much with your hand, thus, but use all gently;
for in the very torrent, tempest, and, as I may say,
the whirlwind of passion, you must acquire and beget
a temperance that may give it smoothness.

O, what a rogue and peasant slave am I!
Is it not monstrous that this player here,
But in a fiction, in a dream of passion,
Could force his soul so to his own conceit
That from her working all his visage wann'd,
Tears in his eyes, distraction in his aspect,
A broken voice, and his whole function suiting
With forms to his conceit? and all for nothing!

Neither a borrower nor a lender be;
For loan oft loses both itself and friend,
And borrowing dulls the edge of husbandry.
This above all: to thine own self be true,
And it must follow, as the night the day,
Thou canst not then be false to any man.

She should have died hereafter;
There would have been a time for such a word.
To-morrow, and to-morrow, and to-morrow,
Creeps in this petty pace from day to day
To the last syllable of recorded time,
And all our yesterdays have lighted fools
The way to dusty death. Out, out, brief candle!
Life's but a walking shadow, a poor player
That struts and frets his hour upon the stage
And then is heard no more: it is a tale
Told by an idiot, full of sound and fury,
Signifying nothing.

Is this a dagger which I see before me,
The handle toward my hand? Come, let me clutch thee.
I have thee not, and yet I see thee still.
Art thou not, fatal vision, sensible
To feeling as to sight? or art thou but
A dagger of the mind, a false creation,
Proceeding from the heat-oppressed brain?
I see thee yet, in form as palpable
As this which now I draw.
Thou marshall'st me the way that I was going;
And such an instrument I was to use.

Double, double toil and trouble;
Fire burn, and cauldron bubble.
Fillet of a fenny snake,
In the cauldron boil and bake;
Eye of newt and toe of frog,
Wool of bat and tongue of dog,
Adder's fork and blind-worm's sting,
Lizard's leg and owlet's wing,
For a charm of powerful trouble,
Like a hell-broth boil and bubble.

Out, damned spot! out, I say! One: two: why,
then, 'tis time to do it. Hell is murky! Fie, my
lord, fie! a soldier, and afeard? What need we
fear who knows it, when none can call our power
to account?

Friends, Romans, countrymen, lend me your ears;
I come to bury Caesar, not to praise him.
The evil that men do lives after them;
The good is oft interred with their bones;
So let it be with Caesar. The noble Brutus
Hath told you Caesar was ambitious:
If it were so, it was a grievous fault,
And grievously hath Caesar answer'd it.
Here, under leave of Brutus and the rest--
For Brutus is an honourable man;
So are they all, all honourable men--
Come I to speak in Caesar's funeral.
He was my friend, faithful and just to me:
But Brutus says he was ambitious;
And Brutus is an honourable man.
He hath brought many captives home to Rome
Whose ransoms did the general coffers fill:
Did this in Caesar seem ambitious?
When that the poor have cried, Caesar hath wept:
Ambition should be made of sterner stuff:
Yet Brutus says he was ambitious;
And Brutus is an honourable man.
You all did see that on the Lupercal
I thrice presented him a kingly crown,
Which he did thrice refuse: was this ambition?
Yet Brutus says he was ambitious;
And, sure, he is an honourable man.
I speak not to disprove what Brutus spoke,
But here I am to speak what I do know.
You all did love him once, not without cause:
What cause withholds you then, to mourn for him?
O judgment! thou art fled to brutish beasts,
And men have lost their reason. Bear with me;
My heart is in the coffin there with Caesar,
And I must pause till it come back to me.

Cowards die many times before their deaths;
The valiant never taste of death but once.
Of all the wonders that I yet have heard,
It seems to me most strange that men should fear;
Seeing that death, a necessary end,
Will come when it will come.

There is a tide in the affairs of men,
Which, taken at the flood, leads on to fortune;
Omitted, all the voyage of their life
Is bound in shallows and in miseries.
On such a full sea are we now afloat;
And we must take the current when it serves,
Or lose our ventures.

Two households, both alike in dignity,
In fair Verona, where we lay our scene,
From ancient grudge break to new mutiny,
Where civil blood makes civil hands unclean.
From forth the fatal loins of these two foes
A pair of star-cross'd lovers take their life;
Whose misadventured piteous overthrows
Do with their death bury their parents' strife.

But, soft! what light through yonder window breaks?
It is the east, and Juliet is the sun.
Arise, fair sun, and kill the envious moon,
Who is already sick and pale with grief,
That thou her maid art far more fair than she.
It is my lady, O, it is my love!
O, that she knew she were!

O Romeo, Romeo! wherefore art thou Romeo?
Deny thy father and refuse thy name;
Or, if thou wilt not, be but sworn my love,
And I'll no longer be a Capulet.
'Tis but thy name that is my enemy;
Thou art thyself, though not a Montague.
What's in a name? that which we call a rose
By any other name would smell as sweet.

This royal throne of kings, this scepter'd isle,
This earth of majesty, this seat of Mars,
This other Eden, demi-paradise,
This fortress built by Nature for herself
Against infection and the hand of war,
This happy breed of men, this little world,
This precious stone set in the silver sea,
Which serves it in the office of a wall,
Or as a moat defensive to a house,
Against the envy of less happier lands,
This blessed plot, this earth, this realm, this England.

Now is the winter of our discontent
Made glorious summer by this sun of York;
And all the clouds that lour'd upon our house
In the deep bosom of the ocean buried.
Now are our brows bound with victorious wreaths;
Our bruised arms hung up for monuments;
Our stern alarums changed to merry meetings,
Our dreadful marches to delightful measures.

Once more unto the breach, dear friends, once more;
Or close the wall up with our English dead.
In peace there's nothing so becomes a man
As modest stillness and humility:
But when the blast of war blows in our ears,
Then imitate the action of the tiger;
Stiffen the sinews, summon up the blood,
Disguise fair nature with hard-favour'd rage;
Then lend the eye a terrible aspect.

This day is called the feast of Crispian:
He that outlives this day, and comes safe home,
Will stand a tip-toe when the day is named,
And rouse him at the name of Crispian.
He that shall live this day, and see old age,
Will yearly on the vigil feast his neighbours,
And say to-morrow is Saint Crispian:
Then will he strip his sleeve and show his scars,
And say these wounds I had on Crispin's day.
Old men forget: yet all shall be forgot,
But he'll remember with advantages
What feats he did that day.
We few, we happy few, we band of brothers;
For he to-day that sheds his blood with me
Shall be my brother; be he ne'er so vile,
This day shall gentle his condition:
And gentlemen in England now a-bed
Shall think themselves accursed they were not here,
And hold their manhoods cheap whiles any speaks
That fought with us upon Saint Crispin's day.

All the world's a stage,
And all the men and women merely players:
They have their exits and their entrances;
And one man in his time plays many parts,
His acts being seven ages. At first the infant,
Mewling and puking in the nurse's arms.
And then the whining school-boy, with his satchel
And shining morning face, creeping like snail
Unwillingly to school. And then the lover,
Sighing like furnace, with a woeful ballad
Made to his mistress' eyebrow. Then a soldier,
Full of strange oaths and bearded like the pard,
Jealous in honour, sudden and quick in quarrel,
Seeking the bubble reputation
Even in the cannon's mouth. And then the justice,
In fair round belly with good capon lined,
With eyes severe and beard of formal cut,
Full of wise saws and modern instances;
And so he plays his part. The sixth age shifts
Into the lean and slipper'd pantaloon,
With spectacles on nose and pouch on side,
His youthful hose, well saved, a world too wide
For his shrunk shank; and his big manly voice,
Turning again toward childish treble, pipes
And whistles in his sound. Last scene of all,
That ends this strange eventful history,
Is second childishness and mere oblivion,
Sans teeth, sans eyes, sans taste, sans everything.

Our revels now are ended. These our actors,
As I foretold you, were all spirits and
Are melted into air, into thin air:
And, like the baseless fabric of this vision,
The cloud-capp'd towers, the gorgeous palaces,
The solemn temples, the great globe itself,
Yea, all which it inherit, shall dissolve
And, like this insubstantial pageant faded,
Leave not a rack behind. We are such stuff
As dreams are made on, and our little life
Is rounded with a sleep.

Full fathom five thy father lies;
Of his bones are coral made;
Those are pearls that were his eyes:
Nothing of him that doth fade
But doth suffer a sea-change
Into something rich and strange.
Sea-nymphs hourly ring his knell:
Ding-dong.
Hark! now I hear them, ding-dong, bell.

The quality of mercy is not strain'd,
It droppeth as the gentle rain from heaven
Upon the place beneath: it is twice blest;
It blesseth him that gives and him that takes:
'Tis mightiest in the mightiest: it becomes
The throned monarch better than his crown;
His sceptre shows the force of temporal power,
The attribute to awe and majesty,
Wherein doth sit the dread and fear of kings;
But mercy is above this sceptred sway;
It is enthroned in the hearts of kings,
It is an attribute to God himself;
And earthly power doth then show likest God's
When mercy seasons justice.

If music be the food of love, play on;
Give me excess of it, that, surfeiting,
The appetite may sicken, and so die.
That strain again! it had a dying fall:
O, it came o'er my ear like the sweet sound,
That breathes upon a bank of violets,
Stealing and giving odour!

Blow, winds, and crack your cheeks! rage! blow!
You cataracts and hurricanoes, spout
Till you have drench'd our steeples, drown'd the cocks!
You sulphurous and thought-executing fires,
Vaunt-couriers to oak-cleaving thunderbolts,
Singe my white head! And thou, all-shaking thunder,
Smite flat the thick rotundity of the world!

The lunatic, the lover and the poet
Are of imagination all compact:
One sees more devils than vast hell can hold,
That is, the madman: the lover, all as frantic,
Sees Helen's beauty in a brow of Egypt:
The poet's eye, in fine frenzy rolling,
Doth glance from heaven to earth, from earth to heaven;
And as imagination bodies forth
The forms of things unknown, the poet's pen
Turns them to shapes and gives to airy nothing
A local habitation and a name.

How many thousand of my poorest subjects
Are at this hour asleep! O sleep, O gentle sleep,
Nature's soft nurse, how have I frighted thee,
That thou no more wilt weigh my eyelids down
And steep my senses in forgetfulness?
Why rather, sleep, liest thou in smoky cribs,
Upon uneasy pallets stretching thee
And hush'd with buzzing night-flies to thy slumber,
Than in the perfumed chambers of the great,
Under the canopies of costly state,
And lull'd with sound of sweetest melody?
Uneasy lies the head that wears a crown.

The barge she sat in, like a burnish'd throne,
Burn'd on the water: the poop was beaten gold;
Purple the sails, and so perfumed that
The winds were love-sick with them; the oars were silver,
Which to the tune of flutes kept stroke, and made
The water which they beat to follow faster,
As amorous of their strokes.

No longer mourn for me when I am dead
Than you shall hear the surly sullen bell
Give warning to the world that I am fled
From this vile world, with vilest worms to dwell:
Nay, if you read this line, remember not
The hand that writ it; for I love you so
That I in your sweet thoughts would be forgot
If thinking on me then should make you woe.
O, if, I say, you look upon this verse
When I perhaps compounded am with clay,
Do not so much as my poor name rehearse.
But let your love even with my life decay,
Lest the wise world should look into your moan
And mock you with me after I am gone.

Full many a glorious morning have I seen
Flatter the mountain-tops with sovereign eye,
Kissing with golden face the meadows green,
Gilding pale streams with heavenly alchemy;
Anon permit the basest clouds to ride
With ugly rack on his celestial face,
And from the forlorn world his visage hide,
Stealing unseen to west with this disgrace:
Even so my sun one early morn did shine
With all triumphant splendour on my brow;
But out, alack! he was but one hour mine;
The region cloud hath mask'd him from me now.
Yet him for this my love no whit disdaineth;
Suns of the world may stain when heaven's sun staineth.

Who steals my purse steals trash; 'tis something, nothing;
'Twas mine, 'tis his, and has been slave to thousands:
But he that filches from me my good name
Robs me of that which not enriches him
And makes me poor indeed.

O, now, for ever
Farewell the tranquil mind! farewell content!
Farewell the plumed troop, and the big wars,
That make ambition virtue! O, farewell!
Farewell the neighing steed, and the shrill trump,
The spirit-stirring drum, the ear-piercing fife,
The royal banner, and all quality,
Pride, pomp and circumstance of glorious war!

Alas, poor Yorick! I knew him, Horatio: a fellow
of infinite jest, of most excellent fancy: he hath
borne me on his back a thousand times; and now, how
abhorred in my imagination it is! my gorge rises at
it. Here hung those lips that I have kissed I know
not how oft. Where be your gibes now? your
gambols? your songs? your flashes of merriment,
that were wont to set the table on a roar?

If it were done when 'tis done, then 'twere well
It were done quickly: if the assassination
Could trammel up the consequence, and catch
With his surcease success; that but this blow
Might be the be-all and the end-all here,
But here, upon this bank and shoal of time,
We'd jump the life to come.

There are more things in heaven and earth, Horatio,
Than are dreamt of in your philosophy.

How sharper than a serpent's tooth it is
To have a thankless child!

Be not afeard; the isle is full of noises,
Sounds and sweet airs, that give delight and hurt not.
Sometimes a thousand twangling instruments
Will hum about mine ears, and sometime voices
That, if I then had waked after long sleep,
Will make me sleep again: and then, in dreaming,
The clouds methought would open and show riches
Ready to drop upon me, that, when I waked,
I cried to dream again.

Be not afraid of greatness: some are born great,
some achieve greatness, and some have greatness
thrust upon them.

Men at some time are masters of their fates:
The fault, dear Brutus, is not in our stars,
But in ourselves, that we are underlings.

O for a Muse of fire, that would ascend
The brightest heaven of invention,
A kingdom for a stage, princes to act
And monarchs to behold the swelling scene!
'''

print(f"语料长度: {len(TEXT):,} 字符")

# 字符级 tokenizer：char -> id（模块 01 的 BPE encode/decode 可直接替换）
chars = sorted(set(TEXT))
vocab_size = len(chars)
cfg['vocab_size'] = vocab_size
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return ''.join(itos[i] for i in ids)

data = torch.tensor(encode(TEXT), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]   # 在"流"层面切分，val 从未被训练

print(f"vocab_size = {vocab_size}")
print('词表:', ''.join(chars).replace('\n', '<NL>'))
print(f"train: {len(train_data):,} tokens | val: {len(val_data):,} tokens")
print(f"ln(vocab_size) = {math.log(vocab_size):.4f}  <- 初始 loss 应接近这个值（第 2 节验证）")


## 2 · MiniGPT：复用模块 02 的架构 + GPT-2 初始化

下面的 `CausalSelfAttention / MLP / Block / MiniGPT` 与模块 02 手写的版本一致（pre-LN、causal mask、weight tying），只额外加了**初始化策略**：

- 所有权重 $\mathcal{N}(0,\,0.02^2)$（GPT-2 [Radford 2019] 的默认）；
- **残差路径上的输出投影**（attn 与 MLP 的 `proj`）的 std 额外乘 $1/\sqrt{2N}$：残差流被 $2N$ 次相加，不缩放则初始方差随深度线性膨胀。

随后做**开训前最便宜的 sanity check**：随机初始化 + 近零 logits 下，softmax 接近均匀分布，交叉熵应当 $\approx \ln V$。
- 显著偏高 → 初始化方差过大；
- 显著偏低 → 几乎必然是标签泄漏（shift 写错 / causal mask 失效，模型看见了答案）。

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, h, T = cfg['n_embd'], cfg['n_head'], cfg['block_size']
        self.h = h
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.proj.IS_RESIDUAL_PROJ = True          # 标记：初始化时做 1/sqrt(2N) 缩放
        self.attn_drop = nn.Dropout(cfg['dropout'])
        self.resid_drop = nn.Dropout(cfg['dropout'])
        self.register_buffer('mask', torch.tril(torch.ones(T, T)).view(1, 1, T, T))

    def forward(self, x):
        B, T, d = x.shape
        hd = d // self.h
        q, k, v = self.qkv(x).split(d, dim=2)
        q = q.view(B, T, self.h, hd).transpose(1, 2)
        k = k.view(B, T, self.h, hd).transpose(1, 2)
        v = v.view(B, T, self.h, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, d)
        return self.resid_drop(self.proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['n_embd']
        self.fc = nn.Linear(d, 4 * d)
        self.proj = nn.Linear(4 * d, d)
        self.proj.IS_RESIDUAL_PROJ = True
        self.drop = nn.Dropout(cfg['dropout'])

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg['n_embd'])
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg['n_embd'])
        self.mlp = MLP(cfg)

    def forward(self, x):          # pre-LN：残差流上只做加法
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['n_embd'])
        self.pos_emb = nn.Embedding(cfg['block_size'], cfg['n_embd'])
        self.drop = nn.Dropout(cfg['dropout'])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg['n_layer'])])
        self.ln_f = nn.LayerNorm(cfg['n_embd'])
        self.lm_head = nn.Linear(cfg['n_embd'], cfg['vocab_size'], bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying（GPT-2 同款）
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            std = 0.02
            if getattr(m, 'IS_RESIDUAL_PROJ', False):
                std = 0.02 / math.sqrt(2 * self.cfg['n_layer'])  # GPT-2 的 1/sqrt(2N) 技巧
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x)
        logits = self.lm_head(self.ln_f(x))        # (B, T, V)
        loss = None
        if targets is not None:                    # T 个位置同时做交叉熵
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


def get_batch(stream, block_size, batch_size):
    # 随机窗口采样：x 是窗口，y 是 x 右移一位（next-token 目标）
    ix = torch.randint(len(stream) - block_size - 1, (batch_size,))
    x = torch.stack([stream[i:i + block_size] for i in ix])
    y = torch.stack([stream[i + 1:i + block_size + 1] for i in ix])
    return x, y


torch.manual_seed(42)
model = MiniGPT(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"参数量: {n_params:,}  (~{n_params / 1e6:.2f}M，weight tying 后)")

# ---- sanity check：初始 loss ≈ ln(V) ----
xb, yb = get_batch(train_data, cfg['block_size'], cfg['batch_size'])
model.eval()
with torch.no_grad():
    _, loss0 = model(xb, yb)
model.train()
expected = math.log(cfg['vocab_size'])
print(f"初始 loss = {loss0.item():.4f} | ln(V) = {expected:.4f}")
assert abs(loss0.item() - expected) < 0.3, '初始 loss 偏离 ln(V)：检查初始化 / causal mask / 标签 shift'
print('✅ sanity check 通过：初始 loss ≈ ln(vocab_size)，没有标签泄漏，初始化健康')

## 3 · 训练循环：AdamW + warmup/cosine + gradient clipping

一个 step 的五个动作：取 batch → 前向 → 反向 → **clip 全局梯度范数到 1.0** → 按当前学习率更新。要点：

- **AdamW** [Loshchilov 2017]：weight decay 与梯度解耦，且**只施加在 2D 矩阵权重上**（bias / LayerNorm / embedding 不衰减）——下面用两个 param group 实现；
- **lr 调度**：前 100 步线性 warmup 到 `max_lr=3e-3`（开局 Adam 二阶矩噪声大），随后 cosine 衰减到 `min_lr = 0.1 * max_lr`（GPT-3 [Brown 2020] 的配方）；
- 每 50 步在 train/val 流上各估一次 loss（多 batch 平均，`eval()` 模式关 dropout）；
- 在 **step 0 / 500 / 1000** 各采样一段文本，亲眼看质量演变。

⏱ 预计耗时：CPU 约 1–3 分钟（共 1000 步）。

In [ ]:
def lr_schedule(step, warmup, max_steps, max_lr, min_lr):
    # 线性 warmup -> cosine 衰减（练习 2 会让你重写它）
    if step < warmup:
        return max_lr * (step + 1) / warmup
    progress = (step - warmup) / (max_steps - warmup)
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))


def estimate_loss(model, stream, iters=8):
    model.eval()
    tot = 0.0
    with torch.no_grad():
        for _ in range(iters):
            xb, yb = get_batch(stream, cfg['block_size'], cfg['batch_size'])
            _, loss = model(xb, yb)
            tot += loss.item()
    model.train()
    return tot / iters


def sample(model, n_chars=300, temperature=0.8, prompt='\n'):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long)
    with torch.no_grad():
        for _ in range(n_chars):
            logits, _ = model(idx[:, -cfg['block_size']:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    model.train()
    return decode(idx[0].tolist())


def train_model(model, tr, va, max_steps=1000, warmup=100,
                max_lr=3e-3, min_lr=3e-4, eval_every=50, sample_steps=()):
    decay = [p for p in model.parameters() if p.dim() >= 2]      # 矩阵权重
    nodecay = [p for p in model.parameters() if p.dim() < 2]     # bias / LN
    opt = torch.optim.AdamW(
        [{'params': decay, 'weight_decay': 0.1},
         {'params': nodecay, 'weight_decay': 0.0}],
        lr=max_lr, betas=(0.9, 0.95))
    hist = {'step': [], 'train': [], 'val': []}
    samples = {}
    for step in range(max_steps + 1):
        if step in sample_steps:
            samples[step] = sample(model)
        if step % eval_every == 0:
            tl, vl = estimate_loss(model, tr), estimate_loss(model, va)
            hist['step'].append(step)
            hist['train'].append(tl)
            hist['val'].append(vl)
            if step % (eval_every * 5) == 0:
                lr_now = lr_schedule(step, warmup, max_steps, max_lr, min_lr)
                print(f"step {step:4d} | lr {lr_now:.2e} | train {tl:.4f} | val {vl:.4f}")
        if step == max_steps:
            break
        lr = lr_schedule(step, warmup, max_steps, max_lr, min_lr)
        for g in opt.param_groups:
            g['lr'] = lr
        xb, yb = get_batch(tr, cfg['block_size'], cfg['batch_size'])
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 全局范数裁剪
        opt.step()
    return hist, samples


t0 = time.time()
hist, samples = train_model(model, train_data, val_data,
                            max_steps=1000, warmup=100, sample_steps=(0, 500, 1000))
print(f"\n训练耗时 {time.time() - t0:.0f}s")
vl = hist['val'][-1]
print(f"最终 val loss = {vl:.4f} -> PPL = {math.exp(vl):.1f} | bits/char = {vl / math.log(2):.2f}")

for s in (0, 500, 1000):
    print('\n' + '=' * 64)
    print(f"--- step {s} 的采样（temperature=0.8）---")
    print(samples[s])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist['step'], hist['train'], label='train')
ax[0].plot(hist['step'], hist['val'], label='val')
ax[0].axhline(math.log(cfg['vocab_size']), ls='--', c='gray', lw=0.8, label='ln(V)')
ax[0].set_xlabel('step'); ax[0].set_ylabel('loss')
ax[0].set_title('train/val loss'); ax[0].legend()

lrs = [lr_schedule(s, 100, 1000, 3e-3, 3e-4) for s in range(1001)]
ax[1].plot(range(1001), lrs)
ax[1].set_xlabel('step'); ax[1].set_ylabel('learning rate')
ax[1].set_title('linear warmup + cosine decay')
plt.tight_layout(); plt.show()

## 4 · 读懂曲线与样本

观察上面的输出（与讲解第 5 节对照）：

- **loss 形状**：从 $\ln V \approx 4.1$ 快速跌落（先学字符频率），随后放缓（学词形、短语，越往后每个 nat 越贵）；
- **样本演变**：step 0 是均匀乱码 → step 500 出现伪英文词形与空格节奏 → step 1000 有局部连贯短语、诗行换行、大写开头——模型学到了"风格"；
- **train/val gap**：~20KB 数据 × 1000 步，每个字符被看上百遍，gap 会逐渐拉开——这正是下一个实验要放大的现象。

## 5 · 过拟合实验：数据砍到 1/10

把 train 流截到前 1/10（约 1.8K tokens），**同样的模型、同样的超参**重训 800 步。预期：train loss 一路向下逼近 0 附近（背诵），val loss 早早触底回升——经典剪刀差。对照讲解第 4 节的"判读三连"。⏱ 约 1–2 分钟。

In [ ]:
torch.manual_seed(7)
small_train = train_data[:len(train_data) // 10]
print(f"小数据集: {len(small_train):,} tokens（原 train 的 1/10）")

model_small = MiniGPT(cfg)
hist_s, _ = train_model(model_small, small_train, val_data,
                        max_steps=800, warmup=80, eval_every=40)

plt.figure(figsize=(6.5, 4))
plt.plot(hist_s['step'], hist_s['train'], label='train (1/10 data)')
plt.plot(hist_s['step'], hist_s['val'], label='val')
plt.axhline(math.log(cfg['vocab_size']), ls='--', c='gray', lw=0.8)
plt.xlabel('step'); plt.ylabel('loss')
plt.title('Overfitting: train/val divergence'); plt.legend(); plt.show()

gap_small = hist_s['val'][-1] - hist_s['train'][-1]
gap_full = hist['val'][-1] - hist['train'][-1]
print(f"1/10 数据 final gap (val - train) = {gap_small:.3f}")
print(f"全量数据 final gap (val - train) = {gap_full:.3f}")
print('解读：val 触底回升的那一步就是早停点；继续训只是在背诵训练集。')

## ✏️ 练习 1：实现 `get_batch(data, block_size, batch_size, rng)`

实现带显式随机源的随机窗口采样（与课程版的区别：用传入的 `torch.Generator` 保证可复现）：

- 用 `torch.randint(..., generator=rng)` 采 `batch_size` 个起点 `ix`（注意上界：起点 + block_size + 1 不能越过流末尾）；
- `x` = 每个起点开始的 `block_size` 个 token；`y` = 整体右移一位；
- 返回形状均为 `(batch_size, block_size)` 的 LongTensor。

提示：`torch.stack` 把一组切片堆成 batch。10 行以内可完成。

In [ ]:
def get_batch_ex(data, block_size, batch_size, rng):
    # TODO: ix = torch.randint(<上界>, (batch_size,), generator=rng)
    # TODO: x = 起点处截 block_size 个 token；y = x 右移一位
    # TODO: return x, y
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
rng = torch.Generator().manual_seed(123)
x, y = get_batch_ex(train_data, 32, 8, rng)
assert x.shape == (8, 32) and y.shape == (8, 32), '形状应为 (batch_size, block_size)'
assert x.dtype == torch.long and y.dtype == torch.long
assert torch.equal(x[:, 1:], y[:, :-1]), 'y 必须是 x 右移一位'

# 同种子可复现
rng2 = torch.Generator().manual_seed(123)
x2, y2 = get_batch_ex(train_data, 32, 8, rng2)
assert torch.equal(x, x2) and torch.equal(y, y2), '相同 Generator 种子应产出相同 batch'

# 边界：多次采样 id 均在词表内（间接验证起点不越界）
rng3 = torch.Generator().manual_seed(7)
for _ in range(20):
    xb, yb = get_batch_ex(train_data, 16, 4, rng3)
    assert int(xb.max()) < vocab_size and int(yb.max()) < vocab_size

print('✅ 练习 1 通过')

## ✏️ 练习 2：实现 `lr_schedule(step, warmup, max_steps, max_lr, min_lr)`

重写训练用的学习率调度（不要直接调用课程版，自己写一遍）：

- `step < warmup`：线性爬升，$\eta = \eta_{max} \cdot (step+1)/warmup$（于是 `step = warmup-1` 时恰好到 `max_lr`）；
- 之后：cosine 从 `max_lr` 衰减到 `min_lr`，进度 $p = (step - warmup)/(max\_steps - warmup)$，$\eta = \eta_{min} + \tfrac12(\eta_{max}-\eta_{min})(1+\cos \pi p)$。

提示：5–8 行。注意 warmup 用 `step+1` 而非 `step`（避免第 0 步 lr 为 0）。

In [ ]:
def lr_schedule_ex(step, warmup, max_steps, max_lr, min_lr):
    # TODO: warmup 段：线性爬升（用 step+1）
    # TODO: cosine 段：min_lr + 0.5*(max_lr-min_lr)*(1+cos(pi*progress))
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测 ----
mx, mn = 3e-3, 3e-4
assert abs(lr_schedule_ex(0, 100, 1000, mx, mn) - mx / 100) < 1e-12, 'step=0 应为 max_lr/warmup'
assert abs(lr_schedule_ex(99, 100, 1000, mx, mn) - mx) < 1e-12, 'warmup 终点应恰好到 max_lr'
assert abs(lr_schedule_ex(1000, 100, 1000, mx, mn) - mn) < 1e-12, 'max_steps 终点应为 min_lr'
assert abs(lr_schedule_ex(550, 100, 1000, mx, mn) - (mx + mn) / 2) < 1e-12, 'cosine 中点应为均值'

# 与课程实现逐点一致
for s in [0, 17, 99, 100, 312, 550, 999, 1000]:
    assert abs(lr_schedule_ex(s, 100, 1000, mx, mn) - lr_schedule(s, 100, 1000, mx, mn)) < 1e-12

print('✅ 练习 2 通过')

## ✏️ 练习 3：实现 `perplexity_from_loss(loss)` 与 `bits_per_char(loss)`

把平均交叉熵（自然对数，单位 nat/char）换算成两个常用报告量：

- $\mathrm{PPL} = e^{\mathcal{L}}$ —— 每步不确定性等价于在多少个等可能选项里均匀猜；
- $\mathrm{bpc} = \mathcal{L} / \ln 2$ —— 每字符比特数（无损压缩率，与 tokenizer 无关的可比单位）。

提示：各 1 行，`math.exp` / `math.log`。两者满足 $\mathrm{PPL} = 2^{\mathrm{bpc}}$。

In [ ]:
def perplexity_from_loss(loss):
    # TODO: e 的 loss 次方
    raise NotImplementedError

def bits_per_char(loss):
    # TODO: nat -> bit
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
assert abs(perplexity_from_loss(0.0) - 1.0) < 1e-12, 'loss=0 -> PPL=1（完美预测）'
assert abs(perplexity_from_loss(math.log(65.0)) - 65.0) < 1e-9, 'loss=ln(65) -> PPL=65（均匀瞎猜 65 选 1）'
assert abs(bits_per_char(math.log(2.0)) - 1.0) < 1e-12, 'loss=ln2 -> 1 bit/char'
assert abs(bits_per_char(4.17) - 4.17 / math.log(2)) < 1e-12

L = 1.234
assert abs(perplexity_from_loss(L) - 2 ** bits_per_char(L)) < 1e-9, '应满足 PPL = 2^bpc'

vl = hist['val'][-1]
print(f"本次训练: val loss {vl:.4f} -> PPL {perplexity_from_loss(vl):.1f} | {bits_per_char(vl):.2f} bits/char")
print('✅ 练习 3 通过')

## 📖 参考答案

先自己做，再对照。每题一个独立 cell。

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def get_batch_ex(data, block_size, batch_size, rng):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,), generator=rng)
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def lr_schedule_ex(step, warmup, max_steps, max_lr, min_lr):
    if step < warmup:
        return max_lr * (step + 1) / warmup
    progress = (step - warmup) / (max_steps - warmup)
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def perplexity_from_loss(loss):
    return math.exp(loss)

def bits_per_char(loss):
    return loss / math.log(2)

## 小结

- **一个目标学出一切**：next-token 交叉熵 = 压缩率；PPL 与 bits/char 是它的两种汇报形式；
- **管线极简**：token 长流 + 随机窗口 `get_batch`，train/val 在流层面切分；
- **训练循环三件套**：AdamW（decoupled weight decay，仅矩阵权重）、warmup+cosine、grad clip 1.0；
- **两个免费的健康检查**：开训前 loss ≈ ln(V)；训练中盯 train/val gap（1/10 数据实验里你已亲眼看到剪刀差）；
- **评测视角**：loss 是体检表不是能力证书——checkpoint 选择、泄漏检查见讲解第 7 节。

➡️ **下一站 · 模块 04《解码策略全手写》**：本章 `sample()` 只用了朴素的温度采样。greedy / temperature / top-k / top-p 各自如何改变生成分布、为何高质量模型也会复读机——全部手写实现并量化对比。

---
## 🎯 真实数据胶囊题：minigpt 该打败的 baseline：真实文本 bigram loss

训练 minigpt 前，先知道“及格线”：一个 bigram 模型在真实 tiny-shakespeare 上的交叉熵。你的 minigpt 若 val loss 低于它才算学到东西。

> 本题为本模块新增的**真实数据**练习：自包含，直接用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, urllib.request, numpy as np
CACHE=os.path.expanduser("~/.llm_internals_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def shakespeare():
    return open(_fetch("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

txt = shakespeare()
chars=sorted(set(txt)); V=len(chars); stoi={c:i for i,c in enumerate(chars)}
ids=np.array([stoi[c] for c in txt]); half=len(ids)//2
train,test=ids[:half],ids[half:]
print(f"真实文本 {len(txt)} 字符, 词表 V={V}")

**练习**：实现 `bigram_val_loss(train, test, V)`：用 train 统计 bigram 转移（加 1 平滑），返回 test 上的平均交叉熵（nats）。这是 minigpt 的及格线。

In [ ]:
def bigram_val_loss(train, test, V):
    # TODO: counts[V,V] 加1平滑 -> 行归一 Q -> -mean(log Q[test[:-1],test[1:]])
    raise NotImplementedError


In [ ]:
# 自测
loss = bigram_val_loss(train, test, V)
assert 0 < loss < np.log(V), "应低于均匀分布的 log(V)"
import math
assert loss < 2.8, "真实文本 bigram char loss 通常 ~2.4-2.6 nats"
print(f"bigram baseline val loss = {loss:.3f} nats (PPL={math.exp(loss):.1f})")
print("=> 你的 minigpt 必须低于这个数才算学到东西")


### 📖 参考答案

In [ ]:
def bigram_val_loss(train, test, V):
    C=np.ones((V,V))
    for a,b in zip(train[:-1],train[1:]): C[a,b]+=1
    Q=C/C.sum(1,keepdims=True)
    return float(-np.log(Q[test[:-1],test[1:]]).mean())
print("✓ 永远先建 baseline，再谈模型有没有用")